# CSPF — reviewer-response analyses (one notebook, resumable)
Answers the strict reviewer's points that need **pair-level predictions or a GPU**. No API; everything is public checkpoints + your files.

| Stage | What it answers | Needs GPU | Time (T4) |
|---|---|---|---|
| **A1** | Training/evaluation overlap: flip-rate (not accuracy) on the 17 datasets no checkpoint saw; per-model own-clean datasets; ranking stability; difference-in-differences for exposed checkpoints | no | 3 min |
| **A2** | Dependence / subpopulations: families compared on *identical reviews*; review-fixed-effects regression with cluster-robust SEs | no | 5 min |
| **A3** | Is the instability still there on *human-verified* meaning-preserving pairs? (225 annotator-1 "yes", 132 consensus "yes") | no | 1 min |
| **B** | Per-model flip rate on NLI-equivalent pairs, and ranking stability (reuses `nli_scores.csv`) | no | 1 min |
| **C** | Ablation: CSPF position-aware insertion vs the *same phrases at random positions* (naive baseline): fluency, NLI equivalence, flips | yes | ~25 min |
| **D** | Consistency training generality: 2 more backbones, held-out phrase variants, independent perturbation generators (typo noise, T5 paraphrase), unseen datasets | yes | ~2-3 h |
| **E** | Overlap with public Hugging Face training sets (exact normalised match) | no | 5 min |

Switch stages with the flags in the next cell. Each stage is wrapped so one failure does not stop the others; all outputs go to Drive and a zip `CSPF_reviewer_bundle.zip` is written at the end (send me that file).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE      = "/content/drive/MyDrive/postdoc-papers/paper2"
FIXED_CSV = f"{BASE}/ALL_MODELS_combined_FIXED.csv"          # per-model predictions (from the alignment-fix notebook)
HUMAN_A1  = f"{BASE}/annotated_output.csv"
HUMAN_A2  = f"{BASE}/annotation_sheet_for_second_annotator_FILLED.csv"
NLI_SCORES= f"{BASE}/results_FIXED/nli/nli_scores.csv"       # from the NLI notebook (stage B only)
OUT       = f"{BASE}/results_FIXED/reviewer"
RUN = dict(A1=True, A2=True, A3=True, B=True, C=True, D=True, E=True)
# datasets that ANY of the 16 checkpoints may have seen (Appendix Table A14): documented or measured overlap
EXPOSED_ANY = {"semeval13","semeval16","semeval17","debate08","hcr"}
import os, subprocess, sys, warnings, traceback, zipfile, glob, time; warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
subprocess.run([sys.executable,"-m","pip","-q","install","sentencepiece","statsmodels"],check=False)
def stage(name):
    def deco(fn):
        def run():
            if not RUN.get(name, False): print(f"[{name}] skipped"); return
            t0=time.time()
            try: fn(); print(f"[{name}] done in {(time.time()-t0)/60:.1f} min")
            except Exception: print(f"[{name}] FAILED:\n", traceback.format_exc())
        return run
    return deco

In [ ]:
# ── load per-pair, per-model flip indicators from the FIXED predictions (CPU, ~3-5 min) ──
import numpy as np, pandas as pd
hdr = pd.read_csv(FIXED_CSV, nrows=0).columns.tolist()
MODELS = sorted({c[:-len("_pert_label")] for c in hdr if c.endswith("_pert_label")})
meta = ["source_dataset","review_idx","original_review","perturbed_review","perturbation_family","severity","sentiment"]
use = meta + [f"{m}_{s}_label" for m in MODELS for s in ("orig","pert")]
parts, Fl = [], []
for ch in pd.read_csv(FIXED_CSV, usecols=use, chunksize=100000, low_memory=False):
    Fl.append(np.stack([(ch[f"{m}_orig_label"].values != ch[f"{m}_pert_label"].values) for m in MODELS], 1).astype(np.int8))
    parts.append(ch[meta])
df = pd.concat(parts, ignore_index=True); F = np.concatenate(Fl); del parts, Fl
df["key"] = df.source_dataset.astype(str)+"::"+df.review_idx.astype(str)
df["pair_id"] = np.arange(len(df)); df["ds"] = df.source_dataset.astype(str).str.lower()
kcodes, kinv = np.unique(df.key.values, return_inverse=True); NK = len(kcodes)
df["mean_flip"] = F.mean(1)
print(len(df), "pairs |", NK, "reviews |", len(MODELS), "models:", MODELS)

In [ ]:
# model -> datasets on which it is (possibly) exposed. CHECK this map against Appendix Table A14 and edit if needed.
import re
def exposure(m):
    s = m.lower()
    if re.search("siebert", s): return {"semeval13","debate08","hcr"}
    if re.search("bertweet|twitter|tweet|cardiff|twroberta", s): return {"semeval13","semeval16","semeval17"}
    return set()
EXPO = {m: exposure(m) for m in MODELS}
for m in MODELS: print(f"{m:35s}", sorted(EXPO[m]) or "-")

In [ ]:
@stage("A1")
def A1():
    clean_any = ~df.ds.isin(EXPOSED_ANY).values
    # (i) flip rate per model: all 22 datasets vs the 17 unexposed datasets
    fr_all = F.mean(0); fr_cl = F[clean_any].mean(0)
    # (ii) "own-clean": each model on the datasets it was NOT exposed to (keeps more data for non-Twitter models)
    fr_own = np.array([F[~df.ds.isin(EXPO[m]).values, i].mean() for i, m in enumerate(MODELS)])
    from scipy.stats import spearmanr, kendalltau
    rng = np.random.default_rng(0)
    def per_key(mask, col): return np.bincount(kinv[mask], weights=F[mask, col], minlength=NK), np.bincount(kinv[mask], minlength=NK)
    S_all = np.stack([per_key(np.ones(len(df), bool), i)[0] for i in range(len(MODELS))], 1); N_all = np.bincount(kinv, minlength=NK)
    S_cl  = np.stack([per_key(clean_any, i)[0] for i in range(len(MODELS))], 1);              N_cl  = np.bincount(kinv[clean_any], minlength=NK)
    rho, tau = [], []
    for _ in range(1000):
        c = np.bincount(rng.integers(0, NK, NK), minlength=NK).astype(float)
        a = (c[:,None]*S_all).sum(0)/(c*N_all).sum(); b = (c[:,None]*S_cl).sum(0)/(c*N_cl).sum()
        rho.append(spearmanr(a, b)[0]); tau.append(kendalltau(a, b)[0])
    out = pd.DataFrame({"model": MODELS, "FR_all22": fr_all.round(4), "FR_17_unexposed": fr_cl.round(4), "FR_own_unexposed": fr_own.round(4),
                        "rank_all22": (-fr_all).argsort().argsort()+1, "rank_17": (-fr_cl).argsort().argsort()+1, "exposed_to": [",".join(sorted(EXPO[m])) for m in MODELS]})
    out.to_csv(f"{OUT}/A1_flip_rate_clean_vs_all.csv", index=False); print(out.to_string(index=False))
    r = dict(spearman=spearmanr(fr_all, fr_cl)[0], spearman_ci=np.percentile(rho,[2.5,97.5]).round(3).tolist(),
             kendall=kendalltau(fr_all, fr_cl)[0], kendall_ci=np.percentile(tau,[2.5,97.5]).round(3).tolist(), spearman_own=spearmanr(fr_all, fr_own)[0])
    print("\nranking agreement all-22 vs 17-unexposed:", r); pd.Series(r).to_csv(f"{OUT}/A1_rank_agreement.csv")
    # (iii) difference-in-differences: exposed checkpoint g vs unexposed controls, on g's exposed datasets E_g vs the clean datasets
    ctrl = [i for i, m in enumerate(MODELS) if not EXPO[m]]; rows = []
    for gi, g in enumerate(MODELS):
        if not EXPO[g]: continue
        E = df.ds.isin(EXPO[g]).values
        SE = np.stack([np.bincount(kinv[E], weights=F[E,i], minlength=NK) for i in range(len(MODELS))], 1); NE = np.bincount(kinv[E], minlength=NK)
        def did(c):
            e = (c[:,None]*SE).sum(0)/(c*NE).sum(); k = (c[:,None]*S_cl).sum(0)/(c*N_cl).sum()
            dg = e[gi]-k[gi]; dc = np.mean([e[i]-k[i] for i in ctrl]); return dg-dc, dg, dc
        est = did(np.ones(NK)); vals = [did(np.bincount(rng.integers(0,NK,NK), minlength=NK).astype(float))[0] for _ in range(500)]
        lo, hi = np.percentile(vals,[2.5,97.5]); rows.append(dict(model=g, exposed=",".join(sorted(EXPO[g])), d_exposed_minus_clean=round(est[1],4), d_controls=round(est[2],4),
                                                                   DiD=round(est[0],4), ci_lo=round(lo,4), ci_hi=round(hi,4), sig=bool(lo>0 or hi<0)))
    D = pd.DataFrame(rows); print("\nDiD (negative = exposed model is MORE stable on its exposed datasets than controls are):\n", D.to_string(index=False)); D.to_csv(f"{OUT}/A1_diff_in_diff.csv", index=False)
A1()

In [ ]:
@stage("A2")
def A2():
    import statsmodels.api as sm
    nrev = df.groupby("perturbation_family").key.nunique()/NK
    UNIV = sorted(nrev[nrev >= 0.99].index); OTHER = [f for f in sorted(df.perturbation_family.unique()) if f not in UNIV]
    print("universal families (apply to ~every review):", UNIV, "\npattern-dependent:", OTHER)
    y = df.mean_flip.values; fam = df.perturbation_family.values; isU = np.isin(fam, UNIV)
    # review-level means: universal baseline and each family
    bU_s = np.bincount(kinv[isU], weights=y[isU], minlength=NK); bU_n = np.bincount(kinv[isU], minlength=NK)
    rng = np.random.default_rng(0); rows = []
    for f in sorted(df.perturbation_family.unique()):
        m = fam == f; s = np.bincount(kinv[m], weights=y[m], minlength=NK); n = np.bincount(kinv[m], minlength=NK)
        elig = (n > 0) & (bU_n > 0); a = s[elig]/n[elig]; b = bU_s[elig]/bU_n[elig]; d = a-b
        bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(1000)]; lo, hi = np.percentile(bs, [2.5, 97.5])
        rows.append(dict(family=f, universal=f in UNIV, eligible_reviews=int(elig.sum()), FR_family_on_eligible=round(a.mean(),4), FR_universal_on_same_reviews=round(b.mean(),4),
                         paired_diff=round(d.mean(),4), ci_lo=round(lo,4), ci_hi=round(hi,4), raw_FR_family=round(y[m].mean(),4), raw_FR_all_universal=round(y[isU].mean(),4)))
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(f"{OUT}/A2_paired_family_vs_universal.csv", index=False)
    # review fixed effects: within-review demeaning, family + severity dummies, cluster-robust SEs by review
    ymean = np.bincount(kinv, weights=y, minlength=NK)/np.bincount(kinv, minlength=NK)
    X = pd.get_dummies(pd.DataFrame({"fam": fam, "sev": df.severity.astype(str).values}), drop_first=False).astype(np.float32)
    ref = "fam_" + UNIV[0]; X = X.drop(columns=[ref, "sev_" + sorted(df.severity.astype(str).unique())[0]])
    Xd = X.values - np.stack([np.bincount(kinv, weights=X[c].values, minlength=NK)/np.bincount(kinv, minlength=NK) for c in X.columns], 1)[kinv]
    res = sm.OLS(y - ymean[kinv], Xd).fit(cov_type="cluster", cov_kwds={"groups": kinv})
    R = pd.DataFrame({"term": X.columns, "coef": res.params.round(4), "se": res.bse.round(4), "p": res.pvalues, "ci_lo": res.conf_int()[:,0].round(4), "ci_hi": res.conf_int()[:,1].round(4)})
    print("\nReview-FE regression of mean_flip (reference family =", UNIV[0], "):\n", R.to_string(index=False)); R.to_csv(f"{OUT}/A2_review_FE_regression.csv", index=False)
    # same, restricted to reviews eligible for the family: does each family still differ from the baseline within the same review?
A2()

In [ ]:
@stage("A3")
def A3():
    a1 = pd.read_csv(HUMAN_A1, encoding="utf-8-sig"); a2 = pd.read_csv(HUMAN_A2, encoding="utf-8-sig")
    a2["meaning_a2"] = a2.meaning_a2.astype(str).str.strip().str.lower()
    h = a1.merge(a2[["sample_id","meaning_a2"]], on="sample_id", how="left")
    h["source_dataset"] = h.source_dataset.astype(str)
    j = h.merge(df[["source_dataset","review_idx","perturbed_review","perturbation_family","pair_id"]].drop_duplicates(["source_dataset","review_idx","perturbed_review","perturbation_family"]),
                on=["source_dataset","review_idx","perturbed_review","perturbation_family"], how="left")
    print(len(h), "audited pairs;", int(j.pair_id.notna().sum()), "matched to the prediction table")
    j = j[j.pair_id.notna()]; Fi = F[j.pair_id.astype(int).values]
    subsets = {"all audited pairs": np.ones(len(j), bool), "annotator 1: meaning = yes": (j.meaning_preserving == "yes").values,
               "annotator 1: not 'no'": (j.meaning_preserving != "no").values,
               "consensus: both annotators 'yes'": ((j.meaning_preserving == "yes") & (j.meaning_a2 == "yes")).values,
               "meaning-altering (annotator 1 'no')": (j.meaning_preserving == "no").values}
    rng = np.random.default_rng(0); rows = []; keys = j.key.values if "key" in j else (j.source_dataset.astype(str)+"::"+j.review_idx.astype(str)).values
    uk, ki = np.unique(keys, return_inverse=True)
    for nm, m in subsets.items():
        if m.sum() == 0: continue
        e = Fi[m].mean(); vals = []
        for _ in range(1000):
            c = np.bincount(rng.integers(0, len(uk), len(uk)), minlength=len(uk))[ki][m]
            vals.append((c[:,None]*Fi[m]).sum()/(c.sum()*Fi.shape[1])) if c.sum() else None
        lo, hi = np.percentile(vals, [2.5, 97.5]); rows.append(dict(subset=nm, pairs=int(m.sum()), mean_FR_16_models=round(e,4), ci_lo=round(lo,4), ci_hi=round(hi,4)))
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(f"{OUT}/A3_FR_on_human_verified.csv", index=False)
    pm = pd.DataFrame({"model": MODELS, **{nm: Fi[m].mean(0).round(4) for nm, m in subsets.items() if m.sum() >= 30}}); pm.to_csv(f"{OUT}/A3_per_model.csv", index=False)
A3()

In [ ]:
@stage("B")
def B():
    from scipy.stats import spearmanr
    S = pd.read_csv(NLI_SCORES); S = S[S.pair_id < len(df)]
    Fs = F[S.pair_id.values]; w = S.w.values; rows = []
    base = (w[:,None]*Fs).sum(0)/w.sum()
    for thr in (0.0, 0.5, 0.9):
        m = (S.nli_bi.values >= thr); fr = (w[m,None]*Fs[m]).sum(0)/w[m].sum()
        rows.append(dict(threshold=thr, pairs=int(m.sum()), mean_FR=round(fr.mean(),4), spearman_with_full_ranking=round(spearmanr(fr, base)[0],3),
                         **{mm: round(v,4) for mm, v in zip(MODELS, fr)}))
    T = pd.DataFrame(rows); print(T.iloc[:, :5].to_string(index=False)); T.to_csv(f"{OUT}/B_per_model_FR_nli_restricted.csv", index=False)
B()

In [ ]:
# ── released corpus (all columns, incl. variant_label / position_strategy) straight from GitHub — needed by stages C, D, E ──
CORPUS_DIR = "/content/CSPF"
if (RUN["C"] or RUN["D"] or RUN["E"]) and not os.path.exists(CORPUS_DIR):
    subprocess.run(["git","clone","--depth","1","https://github.com/hadis-1/CSPF",CORPUS_DIR], check=True)
def load_corpus(cols=None):
    fs = sorted(glob.glob(f"{CORPUS_DIR}/datasets/*_perturbed.csv")); assert fs, "corpus files not found"
    d = pd.concat([pd.read_csv(f, usecols=cols, low_memory=False) for f in fs], ignore_index=True)
    d["key"] = d.source_dataset.astype(str)+"::"+d.review_idx.astype(str); return d

In [ ]:
# ── shared helpers for the GPU stages: NLI (both directions), GPT-2 perplexity, sentiment predictors ──
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
dev = "cuda" if torch.cuda.is_available() else "cpu"; print("device:", dev)
_NLI = {}
@torch.no_grad()
def nli_bi(orig, pert, bs=64, name="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"):
    if "m" not in _NLI:
        _NLI["t"] = AutoTokenizer.from_pretrained(name); _NLI["m"] = AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval().half() if dev=="cuda" else AutoModelForSequenceClassification.from_pretrained(name).eval()
        _NLI["e"] = {v.lower(): k for k, v in _NLI["m"].config.id2label.items()}["entailment"]
    t, m, e = _NLI["t"], _NLI["m"], _NLI["e"]
    def run(P, H):
        out = np.zeros(len(P), dtype=np.float32); order = np.argsort([len(a)+len(b) for a, b in zip(P, H)])
        for i in range(0, len(order), bs):
            ix = order[i:i+bs]; enc = t([P[j] for j in ix], [H[j] for j in ix], padding=True, truncation=True, max_length=160, return_tensors="pt").to(dev)
            out[ix] = torch.softmax(m(**enc).logits.float(), -1)[:, e].cpu().numpy()
        return out
    return np.minimum(run(list(orig), list(pert)), run(list(pert), list(orig)))
@torch.no_grad()
def gpt2_ppl(texts, bs=32):
    if "g" not in _NLI:
        _NLI["gt"] = AutoTokenizer.from_pretrained("gpt2"); _NLI["gt"].pad_token = _NLI["gt"].eos_token
        _NLI["g"] = AutoModelForCausalLM.from_pretrained("gpt2").to(dev).eval()
    t, g = _NLI["gt"], _NLI["g"]; out = np.zeros(len(texts)); order = np.argsort([len(x) for x in texts])
    for i in range(0, len(order), bs):
        ix = order[i:i+bs]; enc = t([texts[j] for j in ix], return_tensors="pt", padding=True, truncation=True, max_length=128).to(dev)
        lg = g(**enc).logits[:, :-1].float(); tg = enc.input_ids[:, 1:]; mk = enc.attention_mask[:, 1:].float()
        nll = torch.nn.functional.cross_entropy(lg.transpose(1, 2), tg, reduction="none")
        out[ix] = torch.exp((nll*mk).sum(1)/mk.sum(1).clamp(min=1)).cpu().numpy()
    return out
SENT_MODELS = ["cardiffnlp/twitter-roberta-base-sentiment-latest", "distilbert-base-uncased-finetuned-sst-2-english", "siebert/sentiment-roberta-large-english"]
@torch.no_grad()
def sent_pred(name, texts, bs=64):
    t = AutoTokenizer.from_pretrained(name); m = AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval()
    if dev == "cuda": m.half()
    out = np.zeros(len(texts), dtype=np.int64); order = np.argsort([len(x) for x in texts])
    for i in range(0, len(order), bs):
        ix = order[i:i+bs]; enc = t([texts[j] for j in ix], padding=True, truncation=True, max_length=128, return_tensors="pt").to(dev)
        out[ix] = m(**enc).logits.argmax(-1).cpu().numpy()
    del m; torch.cuda.empty_cache(); return out

In [ ]:
# ═══ STAGE C — ablation: is CSPF's *position-aware* insertion different from naive insertion of the same phrases at random positions? ═══
import difflib, re
N_PER_FAMILY = 2500; ABL_FAMILIES = ["P01_hedging", "P04_sarcasm"]       # the two insertion-based families
def _norm(w): return re.sub(r"[^\w@#]", "", w.lower())
def inserted_span(o, p):
    """if p = o with ONE contiguous word span inserted (case/punctuation ignored) return (position, span words), else None"""
    ow, pw = o.split(), p.split(); a, b = [_norm(w) for w in ow], [_norm(w) for w in pw]
    ops = [op for op in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes() if op[0] != "equal"]
    if len(ops) == 1 and ops[0][0] == "insert": return ops[0][1], pw[ops[0][3]:ops[0][4]]
    return None
@stage("C")
def C_stage():
    rng = np.random.default_rng(0)
    d = load_corpus(["review_idx","source_dataset","original_review","perturbed_review","perturbation_family","position_strategy","variant_label"])
    rows = []
    for f in ABL_FAMILIES:
        sub = d[d.perturbation_family == f]; sub = sub.sample(min(len(sub), N_PER_FAMILY*3), random_state=0)
        for r in sub.itertuples():
            res = inserted_span(r.original_review, r.perturbed_review)
            if res is None: continue
            pos, span = res; ow = r.original_review.split(); cand = [i for i in range(len(ow)+1) if i != pos]
            if not cand: continue
            q = int(rng.choice(cand)); naive = " ".join(ow[:q] + span + ow[q:])
            rows.append(dict(family=f, strategy=r.position_strategy, orig=r.original_review, cspf=r.perturbed_review, naive=naive, pos_cspf=pos/ max(len(ow),1), pos_naive=q/max(len(ow),1)))
            if sum(1 for x in rows if x["family"] == f) >= N_PER_FAMILY: break
    P = pd.DataFrame(rows); print(P.family.value_counts().to_dict(), "pure-insertion pairs"); assert len(P) > 200
    for v in ("cspf", "naive"):
        P["nli_"+v] = nli_bi(P.orig, P[v]); P["pplr_"+v] = gpt2_ppl(P[v].tolist())/gpt2_ppl(P.orig.tolist())
    for nm in SENT_MODELS:
        po = sent_pred(nm, P.orig.tolist())
        for v in ("cspf", "naive"): P[f"flip_{v}_{nm.split('/')[-1]}"] = (sent_pred(nm, P[v].tolist()) != po).astype(int)
    P.to_csv(f"{OUT}/C_ablation_pairs.csv", index=False)
    fl = [c for c in P.columns if c.startswith("flip_cspf")]; rows = []
    for f in ABL_FAMILIES + ["both"]:
        s = P if f == "both" else P[P.family == f]
        for lab, fn in [("NLI-bi (higher = more equivalent)", lambda v: s["nli_"+v]), ("share NLI-bi >= 0.9", lambda v: (s["nli_"+v] >= .9).astype(float)),
                        ("GPT-2 perplexity ratio (lower = more fluent)", lambda v: s["pplr_"+v]), ("flip rate (mean of 3 models)", lambda v: s[[c.replace("cspf", v) for c in fl]].mean(1))]:
            a, b = fn("cspf").values, fn("naive").values; dd = a-b; bs = [dd[rng.integers(0, len(dd), len(dd))].mean() for _ in range(1000)]
            rows.append(dict(family=f, n=len(s), metric=lab, CSPF=round(a.mean(), 4), naive_random_position=round(b.mean(), 4), diff=round(dd.mean(), 4), ci_lo=round(np.percentile(bs, 2.5), 4), ci_hi=round(np.percentile(bs, 97.5), 4)))
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(f"{OUT}/C_ablation_summary.csv", index=False)
C_stage()

In [ ]:
# ═══ STAGE D — generality of consistency training ═══
# Settings (all fixed in advance). Each run = (split mode, backbone, method, seed); resumable.
D_BACKBONES = ["bert-base-uncased", "roberta-base"]        # add "distilroberta-base" to re-check the original backbone under the new protocol
D_METHODS   = ["erm", "aug_all", "cr_all"]
D_SEEDS     = [0, 1]
D_MODES     = ["review", "dataset"]                        # review: unseen reviews | dataset: unseen DATASETS (4 datasets never seen in training)
HELD_DS     = ["sanders", "hcr", "VADER", "Target-dependent"]   # used in mode "dataset" as the test datasets (tweets from different collections)
EPOCHS, BS, LR, MAXLEN, WARMUP, LAMBDA = 2, 32, 2e-5, 96, 0.06, 1.0
EVAL_CAP, N_PARA, SPLIT_SEED = 4000, 1500, 0

In [ ]:
@stage("D")
def D_stage():
    import torch.nn.functional as Fn, random
    from transformers import get_linear_schedule_with_warmup
    cols = ["review_idx","source_dataset","original_review","perturbed_review","sentiment","perturbation_family","variant_label"]
    d = load_corpus(cols); d["y"] = (d.sentiment == "positive").astype(np.int64)
    d["vh"] = d.variant_label.astype(str).map(lambda s: int(__import__("hashlib").md5(s.encode()).hexdigest(), 16) % 2)   # 0 = "seen" phrase variants, 1 = held-out variants
    FAMS = sorted(d.perturbation_family.unique())
    U = d.drop_duplicates("key")[["key","source_dataset","original_review","y"]].reset_index(drop=True)
    def noise(t, rs, p=0.12):     # independent perturbation generator #1: typographical noise (swap / drop / double a character in ~12% of words)
        out = []
        for w in t.split():
            if len(w) > 3 and rs.random() < p and not w.startswith(("@","#","http")):
                i = int(rs.integers(1, len(w)-1)); op = int(rs.integers(3))
                w = w[:i]+w[i+1]+w[i]+w[i+2:] if op == 0 else (w[:i]+w[i+1:] if op == 1 else w[:i]+w[i]+w[i:])
            out.append(w)
        return " ".join(out)
    para_cache = f"{OUT}/D_t5_paraphrases.csv"
    def paraphrases(keys_texts):
        if os.path.exists(para_cache): return pd.read_csv(para_cache)
        from transformers import AutoModelForSeq2SeqLM
        nm = "humarin/chatgpt_paraphraser_on_T5_base"; t = AutoTokenizer.from_pretrained(nm); m = AutoModelForSeq2SeqLM.from_pretrained(nm).to(dev).eval()
        outs = []
        for i in range(0, len(keys_texts), 32):
            b = keys_texts[i:i+32]; enc = t([f"paraphrase: {x[1]}" for x in b], return_tensors="pt", padding=True, truncation=True, max_length=128).to(dev)
            with torch.no_grad(): g = m.generate(**enc, num_beams=4, max_length=128, repetition_penalty=1.2, no_repeat_ngram_size=3)
            outs += t.batch_decode(g, skip_special_tokens=True)
        P = pd.DataFrame({"key": [x[0] for x in keys_texts], "orig": [x[1] for x in keys_texts], "para": outs})
        P["nli"] = nli_bi(P.orig, P.para); P.to_csv(para_cache, index=False); del m; torch.cuda.empty_cache(); return P

    for mode in D_MODES:
        # ---- split ----
        rng = np.random.default_rng(SPLIT_SEED)
        if mode == "review":
            perm = rng.permutation(len(U)); n = len(U); U["split"] = "train"; U.loc[perm[:int(.2*n)], "split"] = "test"; U.loc[perm[int(.2*n):int(.25*n)], "split"] = "dev"
        else:
            held = U.source_dataset.astype(str).isin(HELD_DS); U["split"] = "train"; U.loc[held, "split"] = "test"
            idx = np.where(~held)[0]; U.loc[rng.choice(idx, int(.05*len(idx)), replace=False), "split"] = "dev"
        sp = U.set_index("key").split; d["split"] = d.key.map(sp)
        print(f"\n=== mode {mode}:", U.split.value_counts().to_dict())
        TR = U[U.split == "train"].reset_index(drop=True); EO = U[U.split == "test"].reset_index(drop=True); DV = U[U.split == "dev"]
        tp = d[d.split == "test"].sample(frac=1, random_state=SPLIT_SEED); EP = tp[tp.groupby("perturbation_family").cumcount() < EVAL_CAP].reset_index(drop=True)
        ft = tp.groupby("perturbation_family").size(); fn_ = EP.groupby("perturbation_family").size(); EP["w"] = EP.perturbation_family.map(ft/fn_).astype(float)
        EP["o_idx"] = EP.key.map(pd.Series(np.arange(len(EO)), index=EO.key)).values
        # training pools: only variants with vh==0 are used for training ("seen phrases"); evaluation reports vh==0 vs vh==1 separately
        trp = d[(d.split == "train") & (d.vh == 0)].reset_index(drop=True); pool = {k: v for k, v in trp.groupby("key").indices.items()}; ptexts = trp.perturbed_review.values
        # independent generators on test originals
        rs = np.random.default_rng(5); EO["noisy"] = [noise(t, rs) for t in EO.original_review]
        sub = EO.sample(min(N_PARA, len(EO)), random_state=0); PR = paraphrases(list(zip(sub.key, sub.original_review)))
        PR = PR[PR.nli >= 0.9].merge(EO[["key","y"]], on="key"); print("  T5 paraphrases kept (NLI-bi >= 0.9):", len(PR))
        for bb in D_BACKBONES:
            tok = AutoTokenizer.from_pretrained(bb)
            enc = lambda tx: tok(list(tx), padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to(dev)
            @torch.no_grad()
            def predict(model, texts, bs=256):
                model.eval(); out = np.zeros(len(texts), dtype=np.int64); order = np.argsort([len(t) for t in texts])
                for i in range(0, len(order), bs):
                    ix = order[i:i+bs]
                    with torch.autocast(dev, dtype=torch.float16, enabled=(dev == "cuda")): out[ix] = model(**enc([texts[j] for j in ix])).logits.argmax(-1).cpu().numpy()
                return out
            sym = lambda lp, lq: 0.5*((lp.exp()*(lp-lq)).sum(-1) + (lq.exp()*(lq-lp)).sum(-1))
            for seed in D_SEEDS:
                for method in D_METHODS:
                    f = f"{OUT}/D/{mode}__{bb.replace('/','_')}__{method}__s{seed}.json"; os.makedirs(os.path.dirname(f), exist_ok=True)
                    if os.path.exists(f): continue
                    t0 = time.time(); torch.manual_seed(seed); rs2 = np.random.default_rng(seed)
                    model = AutoModelForSequenceClassification.from_pretrained(bb, num_labels=2).to(dev)
                    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01); steps = EPOCHS*int(np.ceil(len(TR)/BS))
                    sch = get_linear_schedule_with_warmup(opt, int(WARMUP*steps), steps); scaler = torch.cuda.amp.GradScaler(enabled=(dev == "cuda"))
                    tx, ys, ks = TR.original_review.values, TR.y.values, TR.key.values
                    for ep in range(EPOCHS):
                        model.train(); order = rs2.permutation(len(TR))
                        for s in range(0, len(order), BS):
                            ix = order[s:s+BS]; x = [tx[i] for i in ix]; y = torch.tensor(ys[ix], device=dev)
                            if method != "erm":
                                pj = [(pool[ks[i]][rs2.integers(len(pool[ks[i]]))] if ks[i] in pool else -1) for i in ix]
                                mk = torch.tensor([j >= 0 for j in pj], device=dev, dtype=torch.float32); xp = [ptexts[j] if j >= 0 else tx[i] for i, j in zip(ix, pj)]
                            with torch.autocast(dev, dtype=torch.float16, enabled=(dev == "cuda")):
                                if method == "erm": loss = Fn.cross_entropy(model(**enc(x)).logits.float(), y)
                                else:
                                    lg = model(**enc(x+xp)).logits.float(); lo, lp = lg[:len(x)], lg[len(x):]
                                    ex = (Fn.cross_entropy(lp, y, reduction="none") if method == "aug_all" else sym(Fn.log_softmax(lo, -1), Fn.log_softmax(lp, -1)))
                                    loss = Fn.cross_entropy(lo, y) + LAMBDA*(ex*mk).sum()/mk.sum().clamp(min=1)
                            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step()
                    po = predict(model, EO.original_review.tolist()); pp = predict(model, EP.perturbed_review.tolist())
                    flip = (po[EP.o_idx.values] != pp).astype(float); w = EP.w.values; vh = EP.vh.values
                    wm = lambda x, m: float((x[m]*w[m]).sum()/w[m].sum())
                    allm = np.ones(len(EP), bool); r = dict(mode=mode, backbone=bb, method=method, seed=seed, acc_orig=float((po == EO.y.values).mean()),
                        acc_dev=float((predict(model, DV.original_review.tolist()) == DV.y.values).mean()),
                        FR=wm(flip, allm), FR_seen_variants=wm(flip, vh == 0), FR_heldout_variants=wm(flip, vh == 1))
                    pn = predict(model, EO.noisy.tolist()); r["FR_typo_noise"] = float((pn != po).mean()); r["acc_typo_noise"] = float((pn == EO.y.values).mean())
                    ix = EO.set_index("key").index.get_indexer(PR.key); pq = predict(model, PR.para.tolist()); r["FR_T5_paraphrase"] = float((pq != po[ix]).mean()); r["n_para"] = len(PR)
                    json = __import__("json"); json.dump(r, open(f, "w")); print(f"[{mode} {bb} {method} s{seed}] acc {r['acc_orig']:.3f} FR {r['FR']:.3f} heldout-variants {r['FR_heldout_variants']:.3f} typo {r['FR_typo_noise']:.3f} T5 {r['FR_T5_paraphrase']:.3f} ({(time.time()-t0)/60:.1f} min)")
                    del model; torch.cuda.empty_cache()
    # ---- summary ----
    import json
    R = pd.DataFrame([json.load(open(x)) for x in glob.glob(f"{OUT}/D/*.json")]); R.to_csv(f"{OUT}/D_all_runs.csv", index=False)
    cols_ = ["acc_orig","FR","FR_seen_variants","FR_heldout_variants","FR_typo_noise","FR_T5_paraphrase"]
    S = R.groupby(["mode","backbone","method"])[cols_].agg(["mean","std"]).round(4); print(S.to_string()); S.to_csv(f"{OUT}/D_summary.csv")
    # paired difference vs erm (same backbone, mode, seed): mean ± sd over seeds
    base = R[R.method == "erm"].set_index(["mode","backbone","seed"])[cols_]; rows = []
    for (mo, bb, me), g in R[R.method != "erm"].groupby(["mode","backbone","method"]):
        dd = g.set_index(["mode","backbone","seed"])[cols_] - base.reindex(g.set_index(["mode","backbone","seed"]).index)
        rows.append(dict(mode=mo, backbone=bb, method=me, seeds=len(g), **{"d_"+c: round(dd[c].mean(), 4) for c in cols_}, **{"sd_"+c: round(dd[c].std(), 4) for c in ["FR","FR_heldout_variants","FR_typo_noise","FR_T5_paraphrase"]}))
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(f"{OUT}/D_delta_vs_erm.csv", index=False)
D_stage()

In [ ]:
# ═══ STAGE E — exact-match overlap of the 22 source datasets with public Hugging Face training sets ═══
@stage("E")
def E_stage():
    subprocess.run([sys.executable,"-m","pip","-q","install","datasets"],check=False)
    from datasets import load_dataset
    def nrm(s): return re.sub(r"\s+", " ", re.sub(r"(@\w+|https?://\S+)", "", str(s).lower())).strip()
    U = load_corpus(["review_idx","source_dataset","original_review"]).drop_duplicates("key"); U["n"] = U.original_review.map(nrm)
    SRC = {"tweet_eval/sentiment": ("tweet_eval", "sentiment", ["train","validation","test"], "text"), "sentiment140": ("stanfordnlp/sentiment140", None, ["train","test"], "text"),
           "sst2": ("stanfordnlp/sst2", None, ["train","validation"], "sentence"), "imdb": ("stanfordnlp/imdb", None, ["train","test"], "text"),
           "tweets_hate_offensive/emotion": ("dair-ai/emotion", None, ["train"], "text"), "tweet_eval/irony": ("tweet_eval", "irony", ["train","validation","test"], "text"),
           "tweet_eval/emotion": ("tweet_eval", "emotion", ["train","validation","test"], "text")}
    rows = []
    for nm, (path, cfg, splits, col) in SRC.items():
        try:
            seen = set()
            for s in splits: seen |= set(nrm(x) for x in load_dataset(path, cfg, split=s)[col]) if cfg else set(nrm(x) for x in load_dataset(path, split=s)[col])
        except Exception as e: print(nm, "unavailable:", str(e)[:80]); continue
        U["hit"] = U.n.isin(seen)
        for ds, g in U.groupby("source_dataset"): rows.append(dict(hf_set=nm, dataset=ds, reviews=len(g), exact_matches=int(g.hit.sum()), pct=round(100*g.hit.mean(), 2)))
        print(nm, "->", int(U.hit.sum()), "matches"); 
    T = pd.DataFrame(rows); T.to_csv(f"{OUT}/E_hf_overlap.csv", index=False); print(T[T.exact_matches > 0].to_string(index=False))
E_stage()

In [ ]:
# ── bundle everything small into one zip ──
with zipfile.ZipFile(f"{OUT}/CSPF_reviewer_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*.csv") + glob.glob(f"{OUT}/D/*.json"): z.write(f, os.path.relpath(f, OUT))
print("send me:", f"{OUT}/CSPF_reviewer_bundle.zip", "—", sorted(os.listdir(OUT)))